# 04 · Sampling Parameters

同一个 prompt，发两次给模型，结果往往不一样——这不是 bug，是设计如此。

模型输出的本质是一个**概率分布**，每次生成都是在这个分布上采样。
`temperature`、`top_p`、`top_k` 这三个参数控制的就是**怎么从这个分布里采样**。

理解它们，你就能精确控制模型输出的「随机性」与「创造力」。

## 1. 模型输出的本质：概率分布

每次生成 token 时，模型并不直接输出一个词，
而是输出词表中**每个 token 的概率**，然后按概率采样。

```
输入: "今天天气"
模型内部输出（logits → softmax）:
  "很好"   → 35%
  "不错"   → 28%
  "晴朗"   → 18%
  "糟糕"   →  8%
  其他词   → 11%
  ────────────────
  采样 → 每次结果可能不同
```

采样参数的作用：**改变这个分布的形状**，从而控制输出的多样性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = ["Arial Unicode MS", "sans-serif"]

# 模拟一个简化的 logits 分布（5 个候选 token）
tokens  = ["很好", "不错", "晴朗", "糟糕", "一般"]
logits  = np.array([2.5, 2.1, 1.5, 0.8, 0.5])

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

probs = softmax(logits)
print("原始概率分布：")
for t, p in zip(tokens, probs):
    bar = "█" * int(p * 40)
    print(f"  {t}  {p:.3f}  {bar}")

## 2. Temperature

Temperature 在 softmax 之前对 logits 做缩放：

$$P(token_i) = \frac{e^{logit_i / T}}{\sum_j e^{logit_j / T}}$$

- **T → 0**：分布极度尖锐，概率最高的 token 几乎必然被选中（贪心）
- **T = 1**：原始分布，不做任何改变
- **T > 1**：分布变平，低概率 token 也有机会被选中（更随机）

In [ ]:
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 4), sharey=True)
colors = ["#E63946", "#F4A261", "#2A9D8F", "#457B9D", "#6D6875"]

for ax, T, color in zip(axes, temperatures, colors):
    p = softmax(logits / T)
    ax.bar(tokens, p, color=color, alpha=0.85)
    ax.set_title(f"T = {T}", fontsize=13)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", labelsize=10)
    for i, v in enumerate(p):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

fig.suptitle("Temperature 对概率分布的影响", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from utils.llm_client import chat

# 相同 prompt，不同 temperature，看输出变化
prompt = "用一句话描述秋天。"

print(f"Prompt: {prompt}\n")
for T in [0.0, 0.5, 1.0, 1.5]:
    # 每个 temperature 跑 2 次，观察稳定性
    results = [chat(prompt, temperature=T, max_tokens=60) for _ in range(2)]
    print(f"── Temperature = {T} ──")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r.strip()}")
    print()

**观察：**
- `T=0`：两次输出几乎相同（确定性输出，适合代码生成、数学计算）
- `T=1`：有变化，但保持合理性
- `T=1.5`：更有创意，但可能出现不寻常的表达

## 3. Top-p（Nucleus Sampling）

Top-p 不直接操作 logits，而是**过滤候选集合**：

1. 把所有 token 按概率从高到低排序
2. 从头累加，直到累计概率 ≥ p
3. 只保留这些 token，重新归一化后采样

```
top_p = 0.9：
  "很好"  35%  ← 累计 35%
  "不错"  28%  ← 累计 63%
  "晴朗"  18%  ← 累计 81%
  "糟糕"   8%  ← 累计 89%
  "一般"   6%  ← 累计 95% > 90% ← 截止
  ────────────────────────────────
  只从前 4 个 token 中采样
```

**优点：** 候选集合大小随分布动态变化——分布集中时候选少，分布分散时候选多。

In [ ]:
def apply_top_p(probs: np.ndarray, p: float) -> np.ndarray:
    """演示 top-p 过滤逻辑（仅用于理解原理）。"""
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumulative = np.cumsum(sorted_probs)

    # 找到累计超过 p 的截止点
    cutoff = np.searchsorted(cumulative, p) + 1

    # 只保留前 cutoff 个
    mask = np.zeros_like(probs)
    mask[sorted_idx[:cutoff]] = probs[sorted_idx[:cutoff]]
    return mask / mask.sum()  # 重新归一化

print(f"{'Top-p':>8} | {'保留 token 数':>14} | 分布")
print("-" * 55)
for p in [0.5, 0.7, 0.9, 0.95, 1.0]:
    filtered = apply_top_p(probs, p)
    n_kept = (filtered > 0).sum()
    kept = [(tokens[i], filtered[i]) for i in range(len(tokens)) if filtered[i] > 0]
    kept_str = ", ".join(f"{t}({v:.2f})" for t, v in kept)
    print(f"{p:>8} | {n_kept:>14} | {kept_str}")

In [ ]:
# 实际 API：对比不同 top_p 的创意程度
prompt = "给一家咖啡店起一个有创意的名字，只给名字，不要解释。"

print(f"Prompt: {prompt}\n")
for p in [0.3, 0.7, 0.95]:
    results = [chat(prompt, temperature=1.0, top_p=p, max_tokens=20) for _ in range(3)]
    print(f"── top_p = {p} ──")
    for r in results:
        print(f"  {r.strip()}")
    print()

## 4. Top-k

Top-k 更简单粗暴：**固定只保留概率最高的 k 个 token**，不管概率分布形状如何。

```
top_k = 3：
  保留："很好"(35%)、"不错"(28%)、"晴朗"(18%)
  丢弃："糟糕"(8%)、"一般"(6%)
```

**缺点：** k 是固定的，无法适应分布的集中程度。
- 分布集中时（如 k=50 里 top-1 已有 90% 概率）：仍然采样 50 个，引入不必要的噪声
- 分布分散时（如 top-50 才覆盖 60%）：截断太早，错过合理候选

In [ ]:
def apply_top_k(probs: np.ndarray, k: int) -> np.ndarray:
    """演示 top-k 过滤逻辑。"""
    top_k_idx = np.argsort(probs)[::-1][:k]
    mask = np.zeros_like(probs)
    mask[top_k_idx] = probs[top_k_idx]
    return mask / mask.sum()

print(f"{'Top-k':>8} | 保留的 token")
print("-" * 50)
for k in [1, 2, 3, 4, 5]:
    filtered = apply_top_k(probs, k)
    kept = [(tokens[i], filtered[i]) for i in range(len(tokens)) if filtered[i] > 0]
    kept_str = ", ".join(f"{t}({v:.2f})" for t, v in kept)
    print(f"{k:>8} | {kept_str}")

## 5. 三者的关系与实战建议

通常同时设置 temperature + top_p，**两个条件都满足才采样**：先用 top_p 过滤候选，再用 temperature 控制分布形状。

In [ ]:
# 不同任务的推荐参数组合
presets = [
    {
        "场景": "代码生成 / 数学计算",
        "temperature": 0.0,
        "top_p": 1.0,
        "说明": "确定性输出，每次结果一致",
        "prompt": "写一个 Python 函数计算斐波那契数列第 n 项。",
    },
    {
        "场景": "信息提取 / 问答",
        "temperature": 0.3,
        "top_p": 0.9,
        "说明": "低随机性，保证准确，允许少量表达变化",
        "prompt": "Python 是哪年发布的？只回答年份。",
    },
    {
        "场景": "通用对话",
        "temperature": 0.7,
        "top_p": 0.95,
        "说明": "平衡创意与连贯性，API 默认值附近",
        "prompt": "给我讲一个关于程序员的笑话。",
    },
    {
        "场景": "创意写作 / 头脑风暴",
        "temperature": 1.2,
        "top_p": 0.95,
        "说明": "高随机性，鼓励新颖表达",
        "prompt": "用一句诗描述人工智能。",
    },
]

for preset in presets:
    print(f"\n{'='*55}")
    print(f"场景: {preset['场景']}")
    print(f"参数: temperature={preset['temperature']}, top_p={preset['top_p']}")
    print(f"说明: {preset['说明']}")
    print(f"Prompt: {preset['prompt']}")
    response = chat(
        preset["prompt"],
        temperature=preset["temperature"],
        top_p=preset["top_p"],
        max_tokens=150,
    )
    print(f"Output: {response.strip()}")

In [ ]:
# 极端参数演示：temperature=2.0 时输出会有多混乱
print("Temperature=2.0（极高随机性）的输出：")
for i in range(3):
    r = chat("今天天气很", temperature=2.0, max_tokens=30)
    print(f"  [{i+1}] {r.strip()}")

## 小结

| 参数 | 作用 | 推荐范围 |
|------|------|----------|
| `temperature` | 缩放 logits，控制分布的尖锐程度 | 0（确定）→ 1.5（创意） |
| `top_p` | 动态过滤候选集，保留累计概率前 p 的 token | 0.9–0.95（通用） |
| `top_k` | 固定保留概率最高的 k 个 token | 通常不单独使用 |

**经验法则：**
- 精确任务（代码、计算、提取）→ `temperature=0`
- 通用对话 → `temperature=0.7, top_p=0.95`
- 创意任务 → `temperature=1.0–1.3, top_p=0.95`
- `temperature` 和 `top_p` 一起调，但**不要同时调 top_k**

---

**Module 1 · 基础概念 完结** 🎉

**下一模块 →** [Module 2: Prompting](../02_prompting/01_zero_few_shot.ipynb)：Zero-shot、Few-shot、Chain-of-Thought